# GeoAtt-PointNet++ — Phase 2: Training + Evaluation

**4 Variant Fair Ablation × 5 Seeds**

| Variant | Train flag | Description |
|---------|------------|-------------|
| no_geom | _(none — default)_ | Baseline: PointNet++ murni, tanpa cabang geometri |
| gam_only | `--use-gam` | PointNet++ + Geometric Attention saja |
| fuse_only | `--use-geom-fusion` | PointNet++ + Geometry Fusion (concat) saja |
| with_geom | `--use-geom` | Full GeoAtt-PointNet++ (GAM + Fusion) |

**Seeds:** 7, 42, 123, 2026, 31337
**Split Seed:** 42 (fixed)

> Hipotesis: GeoAtt memberikan dampak signifikan pada identifikasi telapak tangan.
> Bukti dihasilkan oleh notebook 02 lewat paired t-test `with_geom` vs `no_geom`.

## 1. Setup — Mount Drive & Install

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("[ENV] PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True")

!pip install -q torch torchvision numpy scikit-learn matplotlib tqdm pandas seaborn

import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/3DCNN")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project: {PROJECT_ROOT}")
print(f"Exists: {PROJECT_ROOT.exists()}")


## 2. Launch TensorBoard

Kalau muncul `TypeError: Failed to fetch`:
1. **Rerun cell ini** — Colab proxy sering belum siap di percobaan pertama (paling umum).
2. Pastikan direktori `runs/` sudah ada (cell di bawah otomatis `mkdir`).
3. Di Chrome: enable third-party cookies untuk `*.googleusercontent.com` (Settings → Privacy → Cookies → "Allow all").
4. Atau lewati cell ini sekarang dan jalankan setelah training mulai mengisi `runs/`.

In [ ]:
import os

RUNS_DIR = str(PROJECT_ROOT / "runs")
os.makedirs(RUNS_DIR, exist_ok=True)
print(f"RUNS_DIR: {RUNS_DIR}")

%load_ext tensorboard
%tensorboard --logdir $RUNS_DIR --reload_interval 30

## 3. Configuration

In [ ]:
# ============================================================
# PHASE SELECTION
# ============================================================
SEEDS = [42]
# SEEDS = [7, 42, 123, 2026, 31337]

VARIANTS = ["no_geom", "gam_only", "fuse_only", "with_geom"]

FLAGS = {
    "no_geom":   "",
    "gam_only":  "--use-gam",
    "fuse_only": "--use-geom-fusion",
    "with_geom": "--use-geom",
}

DATA_DIR = PROJECT_ROOT / "dataset"
RUNS_DIR = PROJECT_ROOT / "runs"
EVAL_DIR = PROJECT_ROOT / "eval_results"

EPOCHS = 100
PATIENCE = 15

# ============================================================
# VRAM-AWARE CONFIG
# ============================================================
import subprocess
import json

try:
    out = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
        text=True
    )
    VRAM_GB = int(out.strip().split('\n')[0]) / 1024
except Exception:
    VRAM_GB = 0

try:
    out_name = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader,nounits'],
        text=True
    )
    GPU_NAME = out_name.strip().split('\n')[0].strip()
except Exception:
    GPU_NAME = "Unknown"

try:
    import torch
    if torch.cuda.is_available():
        cc_major, cc_minor = torch.cuda.get_device_capability()
        COMPUTE_CAPABILITY = f"{cc_major}.{cc_minor}"
        CC_FLOAT = float(f"{cc_major}.{cc_minor}")
    else:
        COMPUTE_CAPABILITY = "N/A"
        CC_FLOAT = 0.0
except Exception:
    COMPUTE_CAPABILITY = "N/A"
    CC_FLOAT = 0.0

IS_H100 = "H100" in GPU_NAME
IS_BLACKWELL = CC_FLOAT >= 10.0

# ============================================================
# VRAM TARGET — push hingga maksimum
# ============================================================
if VRAM_GB >= 90:
    TARGET_PEAK_GB = 93.0
    MAX_PEAK_GB    = VRAM_GB - 1.5
else:
    TARGET_PEAK_GB = max(VRAM_GB - 5.0, VRAM_GB * 0.93)
    MAX_PEAK_GB    = max(VRAM_GB - 2.5, VRAM_GB * 0.97)

if VRAM_GB >= 90 or (IS_H100 and VRAM_GB >= 85):
    VARIANT_CONFIG = {
        "no_geom":   {"bs": 640, "npts": 8192, "label": "96GB push max"},
        "gam_only":  {"bs": 512, "npts": 8192, "label": "96GB push max"},
        "fuse_only": {"bs": 448, "npts": 8192, "label": "96GB push max"},
        "with_geom": {"bs": 384, "npts": 8192, "label": "96GB push max"},
    }
elif IS_H100 and VRAM_GB >= 75:
    VARIANT_CONFIG = {
        "no_geom":   {"bs": 448, "npts": 8192, "label": "H100 80GB"},
        "gam_only":  {"bs": 384, "npts": 8192, "label": "H100 80GB"},
        "fuse_only": {"bs": 320, "npts": 8192, "label": "H100 80GB"},
        "with_geom": {"bs": 256, "npts": 8192, "label": "H100 80GB"},
    }
elif VRAM_GB >= 75:
    VARIANT_CONFIG = {
        "no_geom":   {"bs": 320, "npts": 8192, "label": "A100 80GB"},
        "gam_only":  {"bs": 256, "npts": 8192, "label": "A100 80GB"},
        "fuse_only": {"bs": 192, "npts": 8192, "label": "A100 80GB"},
        "with_geom": {"bs": 192, "npts": 8192, "label": "A100 80GB"},
    }
else:
    VARIANT_CONFIG = None

BATCH_SIZE_OVERRIDE = None
N_POINTS_OVERRIDE   = None

# Auto-load hasil auto-tune sebelumnya (kalau ada)
AUTOTUNE_CACHE = PROJECT_ROOT / "auto_tune_config.json"
if AUTOTUNE_CACHE.exists():
    try:
        with open(AUTOTUNE_CACHE, "r") as f:
            cached = json.load(f)
        if all(v in cached for v in VARIANTS):
            VARIANT_CONFIG = cached
            print(f"[Auto-tune] Loaded cached config from {AUTOTUNE_CACHE}")
    except Exception:
        pass

# ============================================================
# MIXED PRECISION
# ============================================================
if CC_FLOAT >= 8.0:
    AMP_MODE = "bf16"
elif CC_FLOAT >= 7.0:
    AMP_MODE = "fp16"
else:
    AMP_MODE = "none"

# ============================================================
# CUDA ALLOCATOR
# ============================================================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
CUDA_ALLOC_CONF = os.environ["PYTORCH_CUDA_ALLOC_CONF"]


def get_variant_config(variant: str):
    if BATCH_SIZE_OVERRIDE is not None or N_POINTS_OVERRIDE is not None:
        return {"bs": BATCH_SIZE_OVERRIDE, "npts": N_POINTS_OVERRIDE}
    if VARIANT_CONFIG is not None:
        return VARIANT_CONFIG.get(variant, {"bs": None, "npts": None})
    return {"bs": None, "npts": None}


print("=== CONFIG ===")
print(f"GPU:        {GPU_NAME} (CC={COMPUTE_CAPABILITY}, Blackwell={IS_BLACKWELL})")
print(f"VRAM:       {VRAM_GB:.1f} GB")
print(f"Target peak: {TARGET_PEAK_GB:.1f} GB  (hard ceiling: {MAX_PEAK_GB:.1f} GB)")
print(f"AMP mode:   {AMP_MODE}")
print(f"Allocator:  {CUDA_ALLOC_CONF}")
print(f"Phase:      {'A (pilot)' if len(SEEDS) == 1 else 'B (full)'}")
print(f"Seeds:      {SEEDS}")
print(f"\nVARIANT_CONFIG:")
for v in VARIANTS:
    cfg = get_variant_config(v)
    src = "manual" if (BATCH_SIZE_OVERRIDE is not None) else (VARIANT_CONFIG[v].get("label", "") if VARIANT_CONFIG else "auto-detect")
    print(f"  {v:12s} → bs={str(cfg['bs']):>4s}, n_points={str(cfg['npts']):>5s}  ({src})")
print(f"Epochs: {EPOCHS}  Patience: {PATIENCE}")

In [ ]:
# ============================================================
# FULL CLEANUP: Hapus SEMUA hasil training, eval, cache, dan config
# Jalankan ini kalau ingin mulai dari nol (misal: ada proses gagal)
# WARNING: Data yang dihapus tidak bisa dikembalikan!
# ============================================================
import shutil

RUN_FULL_CLEANUP = True   # ← Ganti ke False untuk skip cleanup

if not RUN_FULL_CLEANUP:
    print("⏭️  FULL CLEANUP skipped (RUN_FULL_CLEANUP = False)")
else:
    total_removed = 0

    # 1. Hapus semua hasil training (runs/)
    if RUNS_DIR.exists():
        for variant_dir in RUNS_DIR.iterdir():
            if variant_dir.is_dir():
                for item in variant_dir.iterdir():
                    if item.is_dir():
                        shutil.rmtree(item)
                        print(f"Removed run: {item}")
                        total_removed += 1

    # 2. Hapus semua hasil evaluasi (eval_results/)
    if EVAL_DIR.exists():
        for variant_dir in EVAL_DIR.iterdir():
            if variant_dir.is_dir():
                for item in variant_dir.iterdir():
                    if item.is_dir():
                        shutil.rmtree(item)
                        print(f"Removed eval: {item}")
                        total_removed += 1

    # 3. Hapus cache dataset scan
    scan_cache = DATA_DIR / ".dataset_scan_cache.pkl"
    if scan_cache.exists():
        scan_cache.unlink()
        print(f"Removed cache: {scan_cache}")
        total_removed += 1

    # 4. Hapus auto-tune config
    autotune_cache = PROJECT_ROOT / "auto_tune_config.json"
    if autotune_cache.exists():
        autotune_cache.unlink()
        print(f"Removed cache: {autotune_cache}")
        total_removed += 1

    # 5. Hapus summary files lama
    for f in ["phase2_summary.json", "training_summary.json"]:
        p = PROJECT_ROOT / f
        if p.exists():
            p.unlink()
            print(f"Removed file: {p}")
            total_removed += 1

    print(f"\n✅ Full cleanup selesai. {total_removed} item dihapus.")
    print("Silakan restart runtime (Runtime → Restart session) lalu jalankan dari Cell 2.")


## 3b. VRAM Probe (Optional but Recommended)

Probe 1 forward+backward per variant dengan konfigurasi saat ini, ukur peak VRAM, dan beri rekomendasi turun/naik. Lebih akurat dari estimasi karena memperhitungkan PyTorch caching + GeoAtt branch overhead.

Jalankan **sebelum training penuh** kalau VRAM > 80 GB dan ingin push maksimal. Skip kalau pakai default konservatif.

In [ ]:
# ============================================================
# VRAM AUTO-TUNE — push bs sedekat mungkin ke TARGET_PEAK_GB
# Adaptive granularity: 16 (explore) → 8 → 4 (fine-tune)
# ============================================================
import torch
import gc
import json
import sys

sys.path.insert(0, str(PROJECT_ROOT))
from models.siamese import SiamesePalmNet
from losses.contrastive import ContrastiveLoss


def _probe_once(variant: str, bs: int, npts: int, geom_dim: int = 14) -> dict:
    """1 forward+backward. Return peak RESERVED GB / OOM info.
    Using max_memory_reserved() because it reflects allocator state + fragmentation,
    which is the real OOM predictor (not just allocated bytes)."""
    use_gam = variant in ("gam_only", "with_geom")
    use_geom_fusion = variant in ("fuse_only", "with_geom")

    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    try:
        model = SiamesePalmNet(
            geom_dim=geom_dim,
            use_geom=(use_gam or use_geom_fusion),
            use_gam=use_gam,
            use_geom_fusion=use_geom_fusion,
        ).cuda()
        criterion = ContrastiveLoss(margin=0.5).cuda()

        pts_a  = torch.randn(bs, npts, 6, device="cuda")
        pts_b  = torch.randn(bs, npts, 6, device="cuda")
        geom_a = torch.randn(bs, geom_dim, device="cuda")
        geom_b = torch.randn(bs, geom_dim, device="cuda")
        labels = torch.randint(0, 2, (bs,), device="cuda").float()

        amp_dtype = torch.bfloat16 if AMP_MODE == "bf16" else (torch.float16 if AMP_MODE == "fp16" else None)
        if amp_dtype is not None:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                emb_a, emb_b, sim = model(pts_a, geom_a, pts_b, geom_b)
                loss = criterion(sim, labels)
        else:
            emb_a, emb_b, sim = model(pts_a, geom_a, pts_b, geom_b)
            loss = criterion(sim, labels)
        loss.backward()
        torch.cuda.synchronize()

        peak_gb = torch.cuda.max_memory_reserved() / 1024**3
        reserved_gb = torch.cuda.max_memory_reserved() / 1024**3
        result = {"ok": True, "peak_gb": peak_gb, "reserved_gb": reserved_gb, "err": None}
    except torch.cuda.OutOfMemoryError:
        result = {"ok": False, "peak_gb": float("inf"), "reserved_gb": float("inf"), "err": "OOM"}
    except Exception as e:
        result = {"ok": False, "peak_gb": -1, "reserved_gb": -1, "err": f"{type(e).__name__}: {str(e).split(chr(10))[0][:80]}"}
    finally:
        for n in ("model","criterion","pts_a","pts_b","geom_a","geom_b","labels","emb_a","emb_b","sim","loss"):
            if n in locals():
                del locals()[n]
        torch.cuda.empty_cache()
        gc.collect()
    return result


def _round_to(bs: int, step: int) -> int:
    return max(step, (bs // step) * step)


def auto_tune_variant(variant: str, seed_bs: int, npts: int,
                       target_gb: float, max_gb: float,
                       max_iters: int = 20) -> dict:
    """Multi-stage: granularity 16 → 8 → 4 untuk push sedekat mungkin ke target."""
    history = []
    sweet_lo = target_gb * 0.95
    sweet_hi = max_gb

    last_ok_bs = None
    last_ok_peak = None
    lo = 16
    hi = None  # OOM ceiling

    # Start dari 16 (bukan 32) supaya tidak terlalu kasar di ujung.
    # Fine-tune sampai step 4 untuk push sedekat mungkin ke batas OOM.
    granularities = [16, 8, 4]
    stage_idx = 0
    step = granularities[stage_idx]

    bs = _round_to(seed_bs, step)

    for it in range(max_iters):
        # Hindari probe duplikat
        if any(h["bs"] == bs for h in history):
            bs += step
            if hi and bs >= hi:
                bs = lo + step
            if bs <= lo:
                break
            continue

        res = _probe_once(variant, bs, npts)
        history.append({"bs": bs, "step": step,
                        **{k: res[k] for k in ("ok","peak_gb","reserved_gb","err")}})

        if not res["ok"]:
            if res["err"] != "OOM":
                return {"bs": last_ok_bs if last_ok_bs else bs, "npts": npts,
                        "peak_gb": last_ok_peak, "history": history,
                        "fatal_err": res["err"]}
            hi = bs
        else:
            peak = res["peak_gb"]
            last_ok_bs = bs
            last_ok_peak = peak
            lo = bs
            if sweet_lo <= peak <= sweet_hi:
                break  # sweet spot

        # Compute next bs
        if hi is None:
            # Belum ketemu OOM — ekspansi
            bs_next = _round_to(bs * 2, step)
        else:
            mid = (lo + hi) // 2
            bs_next = _round_to(mid, step)

            # Kalau stuck (tidak bisa naik dengan step saat ini), drop ke granularity lebih halus
            if bs_next <= lo or bs_next >= hi:
                stage_idx += 1
                if stage_idx >= len(granularities):
                    break  # sudah granularity terhalus, stop
                step = granularities[stage_idx]
                bs_next = lo + step
                if bs_next >= hi:
                    break

        if bs_next == bs or bs_next < 8:
            break
        bs = bs_next

    return {
        "bs": last_ok_bs if last_ok_bs else 16,
        "npts": npts,
        "peak_gb": last_ok_peak,
        "history": history,
    }


def auto_tune_npts(variant: str, bs: int, npts_seed: int,
                     target_gb: float, max_gb: float,
                     step: int = 1024, max_iters: int = 10) -> dict:
    """Stage 2: dengan bs fixed, naikkan npts untuk push peak mendekati target."""
    history = []
    npts = npts_seed
    best_npts = npts_seed
    best_peak = None

    for it in range(max_iters):
        res = _probe_once(variant, bs, npts)
        history.append({"bs": bs, "npts": npts, "step": step,
                        **{k: res[k] for k in ("ok","peak_gb","reserved_gb","err")}})

        if not res["ok"]:
            break

        peak = res["peak_gb"]
        best_npts = npts
        best_peak = peak

        if peak >= target_gb * 0.98:  # cukup dekat target
            break

        npts += step
        if npts > 32768:
            break

    return {
        "npts": best_npts,
        "peak_gb": best_peak,
        "history": history,
    }


print(f"=== VRAM AUTO-TUNE — Target {TARGET_PEAK_GB:.1f} GB / Max {MAX_PEAK_GB:.1f} GB ===")
print(f"GPU: {GPU_NAME} ({VRAM_GB:.1f} GB) | AMP: {AMP_MODE}")
print(f"Sweet spot: {TARGET_PEAK_GB*0.95:.1f}-{MAX_PEAK_GB:.1f} GB peak")
print(f"Granularity stages: 16 → 8 → 4\n")

tuned_config = {}
for variant in VARIANTS:
    cfg = get_variant_config(variant)
    seed_bs = cfg.get("bs") or 256
    npts    = cfg.get("npts") or 8192

    # ── Stage 1: Tune bs ──
    print(f"\n[{variant}] Stage 1 — tune bs | seed bs={seed_bs}, npts={npts}")
    result_bs = auto_tune_variant(variant, seed_bs, npts, TARGET_PEAK_GB, MAX_PEAK_GB)
    bs_opt = result_bs["bs"]
    for h in result_bs["history"]:
        status = "OK" if h["ok"] else f"FAIL ({h['err']})"
        peak_s = f"{h['peak_gb']:6.1f}G" if h["ok"] else "   --"
        print(f"   bs={h['bs']:>4d} (step={h['step']:>2d}) → peak={peak_s}  {status}")
    if result_bs.get("fatal_err"):
        print(f"   ⚠️  Fatal error: {result_bs['fatal_err']}")
    if result_bs["peak_gb"]:
        util_pct = 100 * result_bs["peak_gb"] / VRAM_GB
        print(f"   → bs={bs_opt}  peak={result_bs['peak_gb']:.1f}G ({util_pct:.1f}% of {VRAM_GB:.0f}G)")

    # ── Stage 2: Tune npts dengan bs fixed ──
    print(f"  Stage 2 — tune npts | bs={bs_opt}, seed npts={npts}")
    result_npts = auto_tune_npts(variant, bs_opt, npts, TARGET_PEAK_GB, MAX_PEAK_GB, step=1024)
    npts_opt = result_npts["npts"]
    for h in result_npts["history"]:
        status = "OK" if h["ok"] else f"FAIL ({h['err']})"
        peak_s = f"{h['peak_gb']:6.1f}G" if h["ok"] else "   --"
        print(f"   npts={h['npts']:>5d} (step={h['step']:>4d}) → peak={peak_s}  {status}")
    if result_npts["peak_gb"]:
        util_pct = 100 * result_npts["peak_gb"] / VRAM_GB
        print(f"   → npts={npts_opt}  peak={result_npts['peak_gb']:.1f}G ({util_pct:.1f}% of {VRAM_GB:.0f}G)")

    tuned_config[variant] = {"bs": bs_opt, "npts": npts_opt, "label": "auto-tuned"}

print("\n=== TUNED VARIANT_CONFIG ===")
for v, cfg in tuned_config.items():
    print(f"  {v:12s} bs={cfg['bs']:>4d}, npts={cfg['npts']:>5d}")

VARIANT_CONFIG = {v: cfg for v, cfg in tuned_config.items()}

# Simpan ke disk supaya survive restart runtime
with open(PROJECT_ROOT / "auto_tune_config.json", "w") as f:
    json.dump(VARIANT_CONFIG, f, indent=2)
print(f"\n✓ VARIANT_CONFIG saved to {PROJECT_ROOT / 'auto_tune_config.json'}")
print("  Cell Config akan auto-load config ini setelah restart.")
print("  Rerun cell-9 (helpers) lalu lanjut training.")

## 4. Helper Functions

In [ ]:
def _bs_npts_args(variant: str = None) -> str:
    if variant:
        cfg = get_variant_config(variant)
    else:
        cfg = {"bs": BATCH_SIZE_OVERRIDE, "npts": N_POINTS_OVERRIDE}
    parts = []
    if cfg.get("bs") is not None:
        parts.append(f"--batch_size {cfg['bs']}")
    if cfg.get("npts") is not None:
        parts.append(f"--n_points {cfg['npts']}")
    return " ".join(parts)


def _amp_arg() -> str:
    return f"--amp {AMP_MODE}" if AMP_MODE != "none" else ""




def _env_prefix() -> str:
    """Env prefix untuk subprocess — inject PYTORCH_CUDA_ALLOC_CONF."""
    return f"PYTORCH_CUDA_ALLOC_CONF={CUDA_ALLOC_CONF}"


def _print_gpu_utilization(tag: str = ""):
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used,memory.total,utilization.gpu',
             '--format=csv,noheader,nounits'],
            text=True
        )
        used, total, util = out.strip().split(', ')
        tag_str = f" [{tag}]" if tag else ""
        print(f"  [GPU{tag_str}] VRAM {used}/{total} MB ({100*int(used)/int(total):.1f}%) | Util {util}%")
    except Exception:
        pass


def run_training(variant: str, seed: int) -> bool:
    out_dir = RUNS_DIR / variant / f"seed_{seed}"
    ckpt = out_dir / "best.pth"
    if ckpt.exists():
        print(f"  SKIP training {variant} seed {seed} (already done)")
        return True

    flag = FLAGS[variant]
    cmd = (
        f"{_env_prefix()} "
        f"python {PROJECT_ROOT / 'train.py'} "
        f"--data_dir {DATA_DIR} "
        f"--output_dir {out_dir} "
        f"--seed {seed} "
        f"--fixed_split "
        f"--epochs {EPOCHS} "
        f"--patience {PATIENCE} "
        f"{_bs_npts_args(variant)} "
        f"{_amp_arg()} "
        f"{flag}"
    )
    print(f"START: {variant} | seed {seed}")
    _print_gpu_utilization("pre-train")
    !{cmd}
    _print_gpu_utilization("post-train")
    return ckpt.exists()


def run_test_eval(variant: str, seed: int) -> bool:
    out_dir = RUNS_DIR / variant / f"seed_{seed}"
    ckpt = out_dir / "best.pth"
    eval_out = EVAL_DIR / variant / f"seed_{seed}"
    result_file = eval_out / "results.json"

    if not ckpt.exists():
        print(f"  SKIP eval {variant} seed {seed} (checkpoint missing)")
        return False
    if result_file.exists():
        print(f"  SKIP eval {variant} seed {seed} (already done)")
        return True

    cmd = (
        f"{_env_prefix()} "
        f"python {PROJECT_ROOT / 'evaluate.py'} "
        f"--data_dir {DATA_DIR} "
        f"--checkpoints {variant}={ckpt} "
        f"--output_dir {eval_out} "
        f"{_bs_npts_args(variant)} "
        f"--save_scores"
    )
    print(f"  EVAL: {variant} | seed {seed}")
    _print_gpu_utilization("pre-eval")
    !{cmd}
    _print_gpu_utilization("post-eval")
    return result_file.exists()


def run_holdout_eval(variant: str, seed: int) -> bool:
    out_dir   = RUNS_DIR / variant / f"seed_{seed}"
    ckpt      = out_dir / "best.pth"
    norm      = out_dir / "normalizer.json"
    eval_out  = EVAL_DIR / variant / f"seed_{seed}" / "holdout"
    result_file = eval_out / "results.json"

    if not ckpt.exists():
        print(f"  SKIP holdout {variant} seed {seed} (checkpoint missing)")
        return False
    if result_file.exists():
        print(f"  SKIP holdout {variant} seed {seed} (already done)")
        return True

    cmd = (
        f"{_env_prefix()} "
        f"python {PROJECT_ROOT / 'evaluate.py'} "
        f"--data_dir {DATA_DIR} "
        f"--checkpoints {variant}={ckpt} "
        f"--normalizer {norm} "
        f"--output_dir {eval_out} "
        f"{_bs_npts_args(variant)} "
        f"--holdout "
        f"--n_holdout_sessions 1 "
        f"--n_probe_frames 3 "
        f"--holdout_seed 42 "
        f"--save_scores"
    )
    print(f"  HOLDOUT: {variant} | seed {seed}")
    _print_gpu_utilization("pre-holdout")
    !{cmd}
    _print_gpu_utilization("post-holdout")
    return result_file.exists()

## 5. no_geom — Training (PointNet++ murni (baseline))

In [ ]:
variant = "no_geom"
for seed in SEEDS:
    run_training(variant, seed)


## 6. no_geom — Test Evaluation

In [ ]:
for seed in SEEDS:
    run_test_eval(variant, seed)


## 7. gam_only — Training (PointNet++ + GAM saja)

In [ ]:
variant = "gam_only"
for seed in SEEDS:
    run_training(variant, seed)


## 8. gam_only — Test Evaluation

In [ ]:
for seed in SEEDS:
    run_test_eval(variant, seed)


## 9. fuse_only — Training (PointNet++ + Fusion saja)

In [ ]:
variant = "fuse_only"
for seed in SEEDS:
    run_training(variant, seed)


## 10. fuse_only — Test Evaluation

In [ ]:
for seed in SEEDS:
    run_test_eval(variant, seed)


## 11. with_geom — Training (Full GeoAtt-PointNet++)

In [ ]:
variant = "with_geom"
for seed in SEEDS:
    run_training(variant, seed)


## 12. with_geom — Test Evaluation

In [ ]:
for seed in SEEDS:
    run_test_eval(variant, seed)


## 13. Holdout Evaluation (Final Exam)

Evaluasi pada **holdout probes** — sesi yang TIDAK pernah masuk training/val/test.
Memanggil `evaluate.py --holdout` yang melakukan `split_holdout_sessions(seed=42)` internally.
Pastikan `--holdout_seed` sama dengan split seed di training (42).

In [ ]:
for variant in VARIANTS:
    for seed in SEEDS:
        run_holdout_eval(variant, seed)


## 14. Completion Summary

In [ ]:
import json

summary = {"training_done": [], "eval_done": [], "holdout_done": [], "scores_npz": []}
for variant in VARIANTS:
    for seed in SEEDS:
        run_out  = RUNS_DIR / variant / f"seed_{seed}"
        eval_out = EVAL_DIR / variant / f"seed_{seed}"
        if (run_out / "best.pth").exists():
            summary["training_done"].append(f"{variant}_seed_{seed}")
        if (eval_out / "results.json").exists():
            summary["eval_done"].append(f"{variant}_seed_{seed}")
        if (eval_out / "holdout" / "results.json").exists():
            summary["holdout_done"].append(f"{variant}_seed_{seed}")
        if (eval_out / f"{variant}_scores.npz").exists():
            summary["scores_npz"].append(f"{variant}_seed_{seed}")

n_total = len(VARIANTS) * len(SEEDS)
print("=== PHASE 2 SUMMARY ===")
print(f"Training:     {len(summary['training_done'])}/{n_total}")
print(f"Test Eval:    {len(summary['eval_done'])}/{n_total}")
print(f"Holdout Eval: {len(summary['holdout_done'])}/{n_total}")
print(f"Scores .npz:  {len(summary['scores_npz'])}/{n_total}")

with open(PROJECT_ROOT / "phase2_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved to {PROJECT_ROOT / 'phase2_summary.json'}")
print()
print("Lanjut: jalankan notebook 02_compare_analyze.ipynb untuk analisis statistik.")
